# Simulation 2b: Mean-Subtracted SVD Analysis

**Hypothese**: Der dominante Singulaerwert in Sim 2 ist ein DC-Offset (gemeinsame Richtung aller Tokens).
Wenn wir diese Richtung explizit abtrennen, koennte die verbleibende Struktur
reichere Subspaces zeigen als die PR=1.0 aus Sim 2 vermuten laesst.

**Analogie** (Audio): DC-Offset entfernen bevor man das Spektrum analysiert.

**Tests**:
1. Exp A2: SVD nach Abzug der dominanten Richtung (nicht nur Mean-Centering!)
2. Exp B2: Rekonstruktion mit/ohne dominante Richtung
3. Exp D2: Parallele Verarbeitung mit separierter dominanter Richtung
4. Vergleich Qwen3-0.6B vs Qwen3-8B (4-bit)

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys, os

sys.path.append(".")
sys.path.append(os.path.join("..", "simulation-1"))
import sim2_helpers as helpers
import sim2_metrics as metrics

# === CONFIG ===
# Auf True setzen um auch 8B zu testen (braucht ~7GB VRAM)
TEST_8B = True

# Prompts (identisch zu Sim 2)
prompts_facts = [
    "The capital of France is Paris.",
    "The chemical symbol for water is H2O.",
    "Einstein is known for the theory of relativity.",
    "The sun is a star in the center of the solar system.",
    "Humans breathe oxygen to survive."
]
prompts_reasoning = [
    "If A > B and B > C, then A must be greater than C.",
    "To solve x + 5 = 10, we subtract 5 from both sides.",
    "The next number in the sequence 2, 4, 8, 16 is 32.",
    "A triangle with three equal sides is called equilateral.",
    "If it rains, the ground gets wet. It is raining, so the ground is wet."
]
prompts_creative = [
    "Once upon a time in a galaxy far, far away,",
    "The neon lights of the city reflected in the puddles,",
    "A giant clockwork dragon roared over the mountain peak,",
    "The secret of the universe was hidden in a small tea cup,",
    "Music filled the air as the stars began to dance."
]
all_prompts = prompts_facts + prompts_reasoning + prompts_creative
print(f"{len(all_prompts)} Prompts geladen.")

15 Prompts geladen.


## Helper: Dominant Direction Analysis

Wichtig: `compute_svd_spectrum` in sim2_helpers zentriert bereits (subtrahiert den Mean).
Das ist standard Mean-Centering. Aber der dominante SV *nach* Centering kann trotzdem
eine quasi-uniforme Richtung sein (z.B. Layer-Norm-Artefakt).

Wir machen hier etwas Staerkeres: die Top-1 SVD-Komponente explizit abtrennen
und den Rest separat analysieren.

In [2]:
def analyze_dominant_direction(acts, label=""):
    """
    Analysiert die dominante Richtung und was nach ihrem Abzug uebrig bleibt.
    acts: [n_tokens, d_model]
    """
    flat = acts.reshape(-1, acts.shape[-1]).float()
    mean = flat.mean(dim=0)
    centered = flat - mean
    
    # Full SVD
    U, S, Vh = torch.linalg.svd(centered, full_matrices=False)
    
    # === Analyse der dominanten Richtung ===
    dominant_dir = Vh[0]  # Top-1 rechter Singulaervektor
    
    # Wie viel Varianz erklaert die dominante Richtung?
    total_var = (S**2).sum()
    dom_var = S[0]**2 / total_var
    
    # Projektion jedes Tokens auf die dominante Richtung
    projections = centered @ dominant_dir  # [n_tokens]
    proj_std = projections.std().item()
    proj_mean = projections.mean().item()  # sollte ~0 sein (zentriert)
    
    # === Residual nach Abzug der dominanten Richtung ===
    # X_residual = X - (X @ v1) * v1^T
    residual = centered - projections.unsqueeze(1) * dominant_dir.unsqueeze(0)
    
    # SVD auf Residual
    U_r, S_r, Vh_r = torch.linalg.svd(residual, full_matrices=False)
    total_var_r = (S_r**2).sum()
    cum_var_r = torch.cumsum(S_r**2, dim=0) / total_var_r
    pr_r = (S_r.sum()**2) / (S_r**2).sum()
    
    # Original PR zum Vergleich
    pr_orig = (S.sum()**2) / (S**2).sum()
    cum_var_orig = torch.cumsum(S**2, dim=0) / total_var
    
    n90_orig = int(torch.searchsorted(cum_var_orig, 0.90).item()) + 1
    n95_orig = int(torch.searchsorted(cum_var_orig, 0.95).item()) + 1
    n90_r = int(torch.searchsorted(cum_var_r, 0.90).item()) + 1
    n95_r = int(torch.searchsorted(cum_var_r, 0.95).item()) + 1
    
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    print(f"Dominant direction: {dom_var:.1%} of total variance")
    print(f"SV ratio top1/top2: {S[0]/S[1]:.1f}:1")
    print(f"")
    print(f"{'Metric':<25} {'Original':>12} {'After removal':>15}")
    print(f"{'-'*52}")
    print(f"{'Participation Ratio':<25} {pr_orig.item():>12.1f} {pr_r.item():>15.1f}")
    print(f"{'Components for 90%':<25} {n90_orig:>12d} {n90_r:>15d}")
    print(f"{'Components for 95%':<25} {n95_orig:>12d} {n95_r:>15d}")
    print(f"{'Top-5 SV (residual)':<25} {' ':>12} {np.array2string(S_r[:5].numpy(), precision=1)}")
    
    return {
        "pr_original": pr_orig.item(),
        "pr_residual": pr_r.item(),
        "dominant_var_share": dom_var.item(),
        "sv_original": S.numpy(),
        "sv_residual": S_r.numpy(),
        "cum_var_original": cum_var_orig.numpy(),
        "cum_var_residual": cum_var_r.numpy(),
        "dominant_direction": dominant_dir,
        "mean": mean,
        "residual": residual,
        "n90_orig": n90_orig,
        "n90_residual": n90_r,
    }

## Part 1: Qwen3-0.6B — Mean-Subtracted Analysis

In [3]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained(
    "Qwen/Qwen3-0.6B",
    device="cuda" if torch.cuda.is_available() else "cpu",
    trust_remote_code=True,
    dtype=torch.float32
)
n_layers = model.cfg.n_layers
d_model = model.cfg.d_model
print(f"Modell: {model.cfg.model_name}, {n_layers} Layer, d_model={d_model}")

layers_to_check = sorted(set([1, n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers - 2]))
print(f"Layer: {layers_to_check}")

acts_clean = helpers.extract_activations(model, all_prompts, layers_to_check)

AttributeError: 'Qwen3Config' object has no attribute 'rope_theta'

In [ ]:
# Exp A2: Dominant Direction Analyse pro Layer
results_06b = {}
for l in layers_to_check:
    results_06b[l] = analyze_dominant_direction(acts_clean[l], f"Qwen3-0.6B Layer {l}")

In [ ]:
# Visualisierung: Original vs. Residual SVD-Spektren
fig, axes = plt.subplots(2, len(layers_to_check), figsize=(5*len(layers_to_check), 10))

for i, l in enumerate(layers_to_check):
    r = results_06b[l]
    
    # Row 1: Cumulative Variance
    axes[0, i].plot(r["cum_var_original"][:100], 'b-', label='Original', alpha=0.7)
    axes[0, i].plot(r["cum_var_residual"][:100], 'r-', label='After removal', alpha=0.7)
    axes[0, i].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)
    axes[0, i].set_title(f"Layer {l} (PR: {r['pr_original']:.1f} -> {r['pr_residual']:.1f})")
    axes[0, i].legend(fontsize=8)
    axes[0, i].grid(True)
    
    # Row 2: SV Spectrum (log)
    axes[1, i].plot(r["sv_original"][:50], 'b-', label='Original', alpha=0.7)
    axes[1, i].plot(r["sv_residual"][:50], 'r-', label='After removal', alpha=0.7)
    axes[1, i].set_yscale('log')
    axes[1, i].set_title(f"Layer {l} SV Spectrum")
    axes[1, i].legend(fontsize=8)
    axes[1, i].grid(True)

axes[0, 0].set_ylabel("Cumulative Variance")
axes[1, 0].set_ylabel("Singular Value (log)")
plt.suptitle("Qwen3-0.6B: Original vs. Dominant-Direction-Removed", fontsize=14)
plt.tight_layout()
plt.show()

## Part 2: Exp D2 — Parallele Verarbeitung MIT separierter dominanter Richtung

Idee: Statt die dominante Richtung in den Subspaces zu lassen (wo sie alles dominiert),
trennen wir sie ab, verarbeiten die Residual-Subspaces parallel, und addieren die
dominante Richtung am Ende wieder drauf.

Das ist wie bei Audio: DC-Offset entfernen, Kanaele getrennt verarbeiten, DC am Ende wieder drauf.

In [ ]:
def parallel_forward_mean_separated(model, activations, layer_idx, k_subspaces, tokens):
    """
    Wie parallel_forward, aber mit separierter dominanter Richtung.
    
    1. Dominante Richtung (Top-1 SV) abtrennen
    2. Residual in k Subspaces zerlegen
    3. Jeden Subspace + dominante Richtung einzeln durch naechsten Layer
    4. Ergebnisse mergen
    """
    original_shape = activations.shape  # [batch, pos, d_model]
    flat = activations.reshape(-1, activations.shape[-1]).float()
    mean = flat.mean(dim=0)
    centered = flat - mean
    
    # SVD
    U, S, Vh = torch.linalg.svd(centered, full_matrices=False)
    
    # Dominante Richtung abtrennen
    dominant = U[:, 0:1] @ torch.diag(S[0:1]) @ Vh[0:1, :]  # Rank-1
    residual = centered - dominant
    
    # Residual-SVD fuer Subspace-Zerlegung
    U_r, S_r, Vh_r = torch.linalg.svd(residual, full_matrices=False)
    d_model = flat.shape[-1]
    block_size = d_model // k_subspaces
    
    output_hook_name = f"blocks.{layer_idx+1}.hook_resid_post"
    hook_name = f"blocks.{layer_idx}.hook_resid_post"
    
    outputs = []
    for i in range(k_subspaces):
        start = i * block_size
        end = (i + 1) * block_size if i < k_subspaces - 1 else d_model
        
        # Subspace-Rekonstruktion: dominante Richtung + dieser Residual-Block
        sub_residual = U_r[:, start:end] @ torch.diag(S_r[start:end]) @ Vh_r[start:end, :]
        sub_act = (dominant + sub_residual + mean).reshape(original_shape).to(model.cfg.device)
        
        def make_hook(act):
            def hook_fn(value, hook):
                return act
            return hook_fn
        
        captured = {}
        def make_capture(store):
            def capture_fn(value, hook):
                store["out"] = value.detach().cpu()
                return value
            return capture_fn
        
        with torch.no_grad():
            model.run_with_hooks(
                tokens,
                fwd_hooks=[
                    (hook_name, make_hook(sub_act)),
                    (output_hook_name, make_capture(captured))
                ]
            )
            outputs.append(captured["out"])
            torch.cuda.empty_cache()
    
    return outputs

print("Helper geladen.")

In [ ]:
# Bester Layer fuer Exp D (hoechste PR nach Dominant-Removal)
best_layer = max(results_06b, key=lambda l: results_06b[l]['pr_residual'])
print(f"Bester Layer nach Dominant-Removal: {best_layer} (PR={results_06b[best_layer]['pr_residual']:.1f})")

# Batch-Aktivierungen fuer Hook-basierte Verarbeitung
acts_batch, tokens = helpers.extract_activations_batched(model, all_prompts, [best_layer])

# Original-Output des naechsten Layers
output_hook = f"blocks.{best_layer+1}.hook_resid_post"
with torch.no_grad():
    _, cache = model.run_with_cache(tokens, names_filter=[output_hook])
    original_next = cache[output_hook].detach().cpu()
    del cache
    torch.cuda.empty_cache()

# Vergleich: Original vs. Mean-Separated parallel forward
k_values = [2, 3, 4, 5]

results_d2_orig = {"k": [], "mse": [], "cos_sim": []}
results_d2_sep = {"k": [], "mse": [], "cos_sim": []}

for k in k_values:
    print(f"\nk={k}:")
    
    # Original (wie Sim 2)
    out_orig = helpers.parallel_forward(model, acts_batch[best_layer], best_layer, k, tokens=tokens)
    merged_orig = helpers.merge_subspace_outputs(out_orig)
    mse_o = torch.mean((merged_orig - original_next)**2).item()
    cos_o = torch.nn.functional.cosine_similarity(
        merged_orig.reshape(-1, merged_orig.shape[-1]),
        original_next.reshape(-1, original_next.shape[-1]), dim=1
    ).mean().item()
    results_d2_orig["k"].append(k)
    results_d2_orig["mse"].append(mse_o)
    results_d2_orig["cos_sim"].append(cos_o)
    
    # Mean-Separated
    out_sep = parallel_forward_mean_separated(model, acts_batch[best_layer], best_layer, k, tokens)
    merged_sep = helpers.merge_subspace_outputs(out_sep)
    mse_s = torch.mean((merged_sep - original_next)**2).item()
    cos_s = torch.nn.functional.cosine_similarity(
        merged_sep.reshape(-1, merged_sep.shape[-1]),
        original_next.reshape(-1, original_next.shape[-1]), dim=1
    ).mean().item()
    results_d2_sep["k"].append(k)
    results_d2_sep["mse"].append(mse_s)
    results_d2_sep["cos_sim"].append(cos_s)
    
    print(f"  Original:       MSE={mse_o:.4f}, CosSim={cos_o:.4f}")
    print(f"  Mean-Separated: MSE={mse_s:.4f}, CosSim={cos_s:.4f}")
    delta = cos_s - cos_o
    print(f"  Delta CosSim:   {delta:+.4f} ({'besser' if delta > 0 else 'schlechter'})")

In [ ]:
# Visualisierung Vergleich
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

x = np.arange(len(k_values))
w = 0.35
ax1.bar(x - w/2, results_d2_orig["cos_sim"], w, label='Original (Sim 2)', color='steelblue')
ax1.bar(x + w/2, results_d2_sep["cos_sim"], w, label='Mean-Separated', color='coral')
ax1.set_xticks(x)
ax1.set_xticklabels([f"k={k}" for k in k_values])
ax1.set_ylabel("Cosine Similarity")
ax1.set_title(f"Parallele Verarbeitung: Original vs Mean-Separated (Layer {best_layer})")
ax1.legend()
ax1.set_ylim(0, 1)
ax1.grid(True, axis='y')

ax2.bar(x - w/2, results_d2_orig["mse"], w, label='Original (Sim 2)', color='steelblue')
ax2.bar(x + w/2, results_d2_sep["mse"], w, label='Mean-Separated', color='coral')
ax2.set_xticks(x)
ax2.set_xticklabels([f"k={k}" for k in k_values])
ax2.set_ylabel("MSE")
ax2.set_title("MSE (lower = better)")
ax2.legend()
ax2.grid(True, axis='y')

plt.tight_layout()
plt.show()

## Part 3: Qwen3-8B (4-bit) — Gleiche Analyse

In [ ]:
import gc

if not TEST_8B:
    print("8B-Test uebersprungen (TEST_8B=False)")
else:
    # Cleanup 0.6B
    del model, acts_clean, acts_batch, tokens
    gc.collect()
    torch.cuda.empty_cache()
    
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
    
    tokenizer_8b = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B", trust_remote_code=True)
    model_8b = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-8B",
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model_8b.eval()
    
    nl = model_8b.config.num_hidden_layers
    dm = model_8b.config.hidden_size
    print(f"Qwen3-8B: {nl} Layer, d_model={dm}")
    print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")
    
    layers_8b = sorted(set([1, nl // 4, nl // 2, 3 * nl // 4, nl - 2]))
    print(f"Layer: {layers_8b}")

In [ ]:
if TEST_8B:
    # Aktivierungen extrahieren (HF-Modell, kein TransformerLens)
    def extract_acts_hf(model_hf, tokenizer, prompts, layers):
        all_acts = {l: [] for l in layers}
        for prompt in prompts:
            captured = {}
            hooks = []
            for l in layers:
                layer_module = model_hf.model.layers[l]
                def make_hook(layer_idx):
                    def hook_fn(module, input, output):
                        hidden = output[0] if isinstance(output, tuple) else output
                        captured[layer_idx] = hidden.detach().cpu().float()
                    return hook_fn
                hooks.append(layer_module.register_forward_hook(make_hook(l)))
            inputs = tokenizer(prompt, return_tensors="pt").to(model_hf.device)
            with torch.no_grad():
                model_hf(**inputs)
            for l in layers:
                if l in captured:
                    act = captured[l]
                    if act.ndim == 3:
                        act = act[0]  # [seq, d_model]
                    all_acts[l].append(act)
            for h in hooks:
                h.remove()
        return {l: torch.cat(all_acts[l], dim=0) for l in layers if all_acts[l]}
    
    acts_8b = extract_acts_hf(model_8b, tokenizer_8b, all_prompts, layers_8b)
    
    results_8b = {}
    for l in layers_8b:
        results_8b[l] = analyze_dominant_direction(acts_8b[l], f"Qwen3-8B Layer {l}")

In [ ]:
if TEST_8B:
    fig, axes = plt.subplots(2, len(layers_8b), figsize=(5*len(layers_8b), 10))
    for i, l in enumerate(layers_8b):
        r = results_8b[l]
        axes[0, i].plot(r["cum_var_original"][:100], 'b-', label='Original', alpha=0.7)
        axes[0, i].plot(r["cum_var_residual"][:100], 'r-', label='After removal', alpha=0.7)
        axes[0, i].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)
        axes[0, i].set_title(f"L{l} (PR: {r['pr_original']:.1f} -> {r['pr_residual']:.1f})")
        axes[0, i].legend(fontsize=8)
        axes[0, i].grid(True)
        axes[1, i].plot(r["sv_original"][:50], 'b-', label='Original', alpha=0.7)
        axes[1, i].plot(r["sv_residual"][:50], 'r-', label='After removal', alpha=0.7)
        axes[1, i].set_yscale('log')
        axes[1, i].set_title(f"L{l} SV Spectrum")
        axes[1, i].legend(fontsize=8)
        axes[1, i].grid(True)
    plt.suptitle("Qwen3-8B: Original vs. Dominant-Direction-Removed", fontsize=14)
    plt.tight_layout()
    plt.show()

## Zusammenfassung

In [ ]:
print("=" * 70)
print("SIMULATION 2b: ERGEBNIS-ZUSAMMENFASSUNG")
print("=" * 70)

print("\n--- Qwen3-0.6B ---")
for l in layers_to_check:
    r = results_06b[l]
    print(f"  Layer {l:2d}: PR {r['pr_original']:.1f} -> {r['pr_residual']:.1f}  "
          f"(dominant: {r['dominant_var_share']:.1%}, 90%: {r['n90_orig']} -> {r['n90_residual']})")

if TEST_8B:
    print("\n--- Qwen3-8B ---")
    for l in layers_8b:
        r = results_8b[l]
        print(f"  Layer {l:2d}: PR {r['pr_original']:.1f} -> {r['pr_residual']:.1f}  "
              f"(dominant: {r['dominant_var_share']:.1%}, 90%: {r['n90_orig']} -> {r['n90_residual']})")

print("\n--- Parallele Verarbeitung (0.6B) ---")
for i, k in enumerate(results_d2_orig["k"]):
    cos_o = results_d2_orig["cos_sim"][i]
    cos_s = results_d2_sep["cos_sim"][i]
    print(f"  k={k}: Original CosSim={cos_o:.4f}, Mean-Sep CosSim={cos_s:.4f} (delta={cos_s-cos_o:+.4f})")

print("\n" + "=" * 70)
best_sep = max(results_d2_sep["cos_sim"])
best_orig = max(results_d2_orig["cos_sim"])
print(f"Bester CosSim Original:       {best_orig:.4f}")
print(f"Bester CosSim Mean-Separated:  {best_sep:.4f}")

if best_sep > 0.8:
    print("\nENTSCHEIDUNG: Mean-Separation loest das Problem! Subspaces sind trennbar")
    print("              wenn die dominante Richtung separat behandelt wird.")
    print("              -> Sim 2 Ergebnis revidiert: POSITIV mit Korrektur")
elif best_sep > best_orig + 0.1:
    print("\nENTSCHEIDUNG: Mean-Separation hilft signifikant, reicht aber nicht.")
    print("              -> Verbesserung, aber Grundproblem bleibt. Weiter zu Sim 3.")
else:
    print("\nENTSCHEIDUNG: Mean-Separation aendert wenig.")
    print("              -> Sim 2 NEGATIV bestaetigt. Die Verflechtung ist nicht")
    print("              nur ein DC-Offset-Artefakt. Weiter zu Sim 3.")
print("=" * 70)